# Bước 7 — Đánh giá mô hình bổ trợ

Nhóm 10, IE313. Vận dụng Bài 06–07; nguồn/trang PDF theo `docs/SPECIFICATION.md` mục 1.3–1.4. Quy ước và phần bổ sung tại `docs/MODELING_METHODS.md`. Notebook đọc baseline giữ nguyên giá dương, chọn biến trên train/fold, xuất kết quả thực; không hạ ngưỡng để ép có hồi quy.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "workspaces/final-project"]
PROJECT = next(p for p in candidates if (p / "src/modeling.py").is_file())
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from src.modeling import run_full_modeling_workflow  # noqa: E402

metadata = run_full_modeling_workflow(
    PROJECT / "data/processed/bangkok_cleaned.csv", PROJECT / "reports/modeling"
)
print(metadata["versions"])

## Chia tập và tính độc lập

70/30 theo listing, seed42; CV 4-fold trên train. ID/host không được làm predictor. Giao host giới hạn suy rộng sang chủ nhà mới. Mỗi fold có quyết định chọn biến riêng trước khi fit imputer/scaler/encoder.

In [ ]:
display(pd.Series(metadata["split_info"]))
assignments = pd.read_csv(PROJECT / "reports/modeling/split_assignments.csv")
display(assignments.groupby(["split", "fold"]).size().rename("n"))

## Chọn biến số và chẩn đoán phân loại

Pearson |r|≥0,30 và p<0,05 là quy ước repository; thống kê chưa làm tròn. **[MỞ RỘNG NGOÀI BÀI HỌC]** Levene center=median chỉ chẩn đoán phương sai, nguồn trong EDA_METHODS. Không gán mọi giả định là sai khi chưa đo; nhóm n=1 được giữ và ghi rõ ANOVA chưa tính. Chính sách thận trọng không dùng ANOVA làm bộ lọc khi điều kiện chưa được xác nhận.

In [ ]:
selection = json.loads((PROJECT / "reports/modeling/feature_selection.json").read_text())
display(pd.DataFrame(selection["full_train"]["numeric_features"]).T)
display(
    pd.DataFrame(selection["full_train"]["categorical_features"]).T[
        ["groups_count", "n_valid", "f_statistic", "p_value", "selected", "reason"]
    ]
)
print("Số biến qua tiêu chí từng fold:", selection["folds_selected_counts"])
print("Số biến full train:", selection["full_train_selected_count"])

## Kết quả đánh giá

DummyRegressor là **bổ sung kỹ thuật theo repository**, dự đoán trung bình y_train riêng từng fold. Nó là mốc so sánh, không thay thế hồi quy có predictor. MSE theo THB²; R² giữ giá trị âm, trả null khi không xác định. CV std dùng ddof=0. Khi có biến hợp lệ, lựa chọn cấu hình bằng CV và chỉ đánh giá holdout sau khi chốt.

In [ ]:
display(pd.DataFrame([metadata["benchmark_results"]]))
display(pd.read_csv(PROJECT / "reports/modeling/model_comparison.csv"))
print(metadata["feasibility_decision"])
print(metadata["conditional_artifacts"])

## Tổng hợp và giới hạn

Diễn giải theo đúng kết quả đang chạy. Chưa chọn được biến theo tiêu chí hiện tại không chứng minh dữ liệu không thể mô hình hóa. Khi không có hồi quy, không xuất file mô hình hay hình chẩn đoán hồi quy. Nhánh phần mềm khả thi được kiểm thử trên dữ liệu tổng hợp; không dùng số liệu kiểm thử trong kết quả đồ án.

In [ ]:
display(Markdown((PROJECT / "reports/modeling/modeling_findings.md").read_text()))